# Loan Default Prediction System (P_137) — Exploratory Data Analysis

Dataset: *Credit Risk Dataset* by laotse on Kaggle (https://www.kaggle.com/datasets/laotse/credit-risk-dataset). This notebook reads the raw CSV, checks quality (missing values, duplicates, impossible values) and looks at how default varies with grade, purpose and the numeric columns. Every chart is saved to `reports/eda/` at 150 dpi.

In [1]:
import matplotlib
matplotlib.use("Agg")  # file output only, no window needed
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EDA_DIR = ROOT / "reports" / "eda"
EDA_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(ROOT / "data" / "raw" / "credit_risk_dataset.csv", encoding="utf-8")
print(df.shape)
df.head()

(32581, 12)


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


## 1. Target: how many loans defaulted?

In [2]:
counts = df["loan_status"].value_counts().sort_index()
print(f"counts {counts.to_dict()}, default rate {df['loan_status'].mean():.3f}")
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Repaid (0)", "Defaulted (1)"], counts.values, color=["#4C72B0", "#C44E52"])
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
ax.set_title("Loans repaid vs defaulted"); ax.set_xlabel("loan_status"); ax.set_ylabel("Number of loans")
plt.tight_layout(); plt.savefig(EDA_DIR / "01_default_rate.png", dpi=150); plt.close()

counts {0: 25473, 1: 7108}, default rate 0.218


## 2. Missing values

In [3]:
missing = df.isna().sum()
missing = missing[missing > 0]
print("missing", missing.to_dict())
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(missing.index, missing.values, color="#DD8452")
for i, v in enumerate(missing.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
ax.set_title("Missing values per column"); ax.set_xlabel("Column"); ax.set_ylabel("Missing rows")
plt.tight_layout(); plt.savefig(EDA_DIR / "02_missing_values.png", dpi=150); plt.close()

missing {'person_emp_length': 895, 'loan_int_rate': 3116}


## 3. Duplicates and outliers

In [4]:
print(f"duplicates: {df.duplicated().sum()} | age > 100: {(df['person_age'] > 100).sum()} | "
      f"employment > 60 years: {(df['person_emp_length'] > 60).sum()}")
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].boxplot(df["person_age"].dropna()); axes[0].set_title("Age"); axes[0].set_ylabel("Years")
axes[1].boxplot(df["person_emp_length"].dropna()); axes[1].set_title("Employment length"); axes[1].set_ylabel("Years")
axes[2].boxplot(df["person_income"].dropna()); axes[2].set_yscale("log")
axes[2].set_title("Income (log scale)"); axes[2].set_ylabel("Annual income")
for a in axes:
    a.set_xticks([])
fig.suptitle("Outliers in age, employment length and income")
plt.tight_layout(); plt.savefig(EDA_DIR / "03_outliers.png", dpi=150); plt.close()

duplicates: 165 | age > 100: 5 | employment > 60 years: 2


## 4. Default rate by the lender's grade

In [5]:
by_grade = (df.groupby("loan_grade")["loan_status"].mean() * 100).round(1).sort_index()
print("default % by grade ", by_grade.to_dict())
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(by_grade.index, by_grade.values, color="#55A868")
for i, v in enumerate(by_grade.values):
    ax.text(i, v, f"{v}", ha="center", va="bottom")
ax.set_title("Default rate by loan grade"); ax.set_xlabel("loan_grade (A = best)"); ax.set_ylabel("Default rate (%)")
plt.tight_layout(); plt.savefig(EDA_DIR / "04_default_by_grade.png", dpi=150); plt.close()

default % by grade  {'A': 10.0, 'B': 16.3, 'C': 20.7, 'D': 59.0, 'E': 64.4, 'F': 70.5, 'G': 98.4}


## 5. Default rate by loan purpose

In [6]:
by_intent = (df.groupby("loan_intent")["loan_status"].mean() * 100).round(1).sort_values()
print("default % by intent", by_intent.to_dict())
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(by_intent.index, by_intent.values, color="#8172B3")
for i, v in enumerate(by_intent.values):
    ax.text(v, i, f" {v}", va="center")
ax.set_title("Default rate by loan purpose"); ax.set_xlabel("Default rate (%)"); ax.set_ylabel("loan_intent")
plt.tight_layout(); plt.savefig(EDA_DIR / "05_default_by_intent.png", dpi=150); plt.close()

default % by intent {'VENTURE': 14.8, 'EDUCATION': 17.2, 'PERSONAL': 19.9, 'HOMEIMPROVEMENT': 26.1, 'MEDICAL': 26.7, 'DEBTCONSOLIDATION': 28.6}


## 6. Correlation between numeric columns

In [7]:
corr = df.select_dtypes("number").corr()
with_target = corr["loan_status"].drop("loan_status").round(2).sort_values(key=abs, ascending=False)
print("correlation with loan_status:", ", ".join(f"{k} {v}" for k, v in with_target.items()))
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation between numeric columns (incl. loan_status)")
plt.tight_layout(); plt.savefig(EDA_DIR / "06_correlation.png", dpi=150); plt.close()

correlation with loan_status: loan_percent_income 0.38, loan_int_rate 0.34, person_income -0.14, loan_amnt 0.11, person_emp_length -0.08, person_age -0.02, cb_person_cred_hist_length -0.02


## Findings

- **Imbalanced target:** 7,108 of 32,581 loans defaulted (21.8%), so accuracy alone would mislead; the models are judged on ROC-AUC, PR-AUC and KS instead.
- **Data quality:** 895 rows miss `person_emp_length` and 3,116 miss `loan_int_rate` (filled later with training medians); 165 exact duplicates, 5 ages over 100 and 2 employment lengths over 60 years are removed in cleaning.
- **Grade is a strong but blunt signal:** default rises from 10.0% in grade A to 20.7% in C and then jumps to 59.0% in D and 98.4% in G; purpose matters less (14.8% VENTURE to 28.6% DEBTCONSOLIDATION).
- **Affordability drives risk:** `loan_percent_income` (0.38) and `loan_int_rate` (0.34) correlate most with default, while higher income (-0.14) and longer employment (-0.08) lower it.